# Sesión 6: Funciones y Operaciones de Conjunto en SQL
## Clase 6 - Fundamentos de Bases de Datos

En esta sesión aprenderemos:
- Funciones agregadas (MIN, MAX, AVG, COUNT)
- GROUP BY para estadísticas por grupo
- DISTINCT para eliminar duplicados
- UNION, INTERSECT, EXCEPT para operaciones de conjunto

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✅ Conexión SQLite establecida para Sesión 6")

## SETUP: Crear tablas de ventas y clientes

In [ ]:
# Crear tabla de ventas
cursor.execute('''
    CREATE TABLE ventas (
        id_venta INTEGER PRIMARY KEY,
        fecha_compra DATE,
        monto_total NUMERIC,
        estado TEXT,
        region TEXT,
        ciudad TEXT,
        producto TEXT
    )
''')

# Datos de ventas con variedad para funciones agregadas
ventas_data = [
    (1, '2024-06-01', 150000, 'Finalizada', 'Norte', 'Arica', 'Laptop'),
    (2, '2024-06-05', 85000, 'Finalizada', 'Norte', 'Iquique', 'Monitor'),
    (3, '2024-06-10', 120000, 'Finalizada', 'Centro', 'Santiago', 'Teclado'),
    (4, '2024-06-15', 95000, 'Cancelada', 'Centro', 'Valparaíso', 'Mouse'),
    (5, '2024-06-20', 180000, 'Finalizada', 'Centro', 'Santiago', 'Laptop'),
    (6, '2024-07-01', 110000, 'Finalizada', 'Sur', 'Concepción', 'Monitor'),
    (7, '2024-07-05', 75000, 'Pendiente', 'Sur', 'Punta Arenas', 'Teclado'),
    (8, '2024-07-10', 165000, 'Finalizada', 'Norte', 'Antofagasta', 'Laptop'),
    (9, '2024-07-15', 90000, 'Finalizada', 'Centro', 'Santiago', 'Monitor'),
    (10, '2024-07-20', 125000, 'Finalizada', 'Sur', 'Puerto Montt', 'Laptop')
]

cursor.executemany(
    'INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?, ?)',
    ventas_data
)

print("✅ Tabla 'ventas' creada con 10 registros")

In [ ]:
# Crear tablas para operaciones de conjunto (Slide 29)
cursor.execute('''
    CREATE TABLE clientes_2023 (
        nombre TEXT,
        correo TEXT,
        ciudad TEXT,
        ingreso_mensual NUMERIC
    )
''')

cursor.execute('''
    CREATE TABLE clientes_2024 (
        nombre TEXT,
        correo TEXT,
        ciudad TEXT,
        ingreso_mensual NUMERIC
    )
''')

# Datos 2023
clientes_2023 = [
    ('Ana', 'ana@mail.com', 'Santiago', 1100000),
    ('Luis', 'luis@mail.com', 'Valparaíso', 950000),
    ('Carlos', 'carlos@mail.com', 'Rancagua', 1050000)
]

# Datos 2024
clientes_2024 = [
    ('Ana', 'ana@mail.com', 'Santiago', 1150000),
    ('Camila', 'camila@mail.com', 'Concepción', 980000),
    ('Carlos', 'carlos@mail.com', 'Rancagua', 1100000)
]

cursor.executemany('INSERT INTO clientes_2023 VALUES (?, ?, ?, ?)', clientes_2023)
cursor.executemany('INSERT INTO clientes_2024 VALUES (?, ?, ?, ?)', clientes_2024)
conn.commit()

print("✅ Tablas 'clientes_2023' y 'clientes_2024' creadas")

## 1. FUNCIONES AGREGADAS (Slides 4-12)

### MIN() y MAX() - Slide 5

In [ ]:
# Slide 5: MIN y MAX
df_min_max = pd.read_sql_query(
    """SELECT 
        MIN(monto_total) AS compra_minima,
        MAX(monto_total) AS compra_maxima,
        MIN(fecha_compra) AS primera_compra,
        MAX(fecha_compra) AS ultima_compra
    FROM ventas""",
    conn
)

print("SLIDE 5: Funciones MIN() y MAX()")
print("\n" + df_min_max.to_string(index=False))

### AVG() - Promedio (Slide 7)

In [ ]:
# Slide 7: AVG
df_avg = pd.read_sql_query(
    """SELECT 
        ROUND(AVG(monto_total), 2) AS promedio_ventas
    FROM ventas
    WHERE estado = 'Finalizada'""",
    conn
)

print("SLIDE 7: Función AVG() - Promedio de Ventas")
print("\nPromedio de ventas finalizadas: $" + str(df_avg.iloc[0, 0]))

### COUNT() - Contar registros (Slide 9)

In [ ]:
# Slide 9: COUNT
print("SLIDE 9: Función COUNT()")
print("="*70)

# Total de ventas
df_count_total = pd.read_sql_query(
    'SELECT COUNT(*) as total_ventas FROM ventas',
    conn
)
print(f"\n1. Total de ventas registradas: {df_count_total.iloc[0, 0]}")

# Productos distintos
df_count_distinct = pd.read_sql_query(
    'SELECT COUNT(DISTINCT producto) as productos_vendidos FROM ventas',
    conn
)
print(f"2. Productos distintos: {df_count_distinct.iloc[0, 0]}")

# Ciudades distintas
df_count_ciudades = pd.read_sql_query(
    'SELECT COUNT(DISTINCT ciudad) as ciudades_activas FROM ventas',
    conn
)
print(f"3. Ciudades activas: {df_count_ciudades.iloc[0, 0]}")

### Combinación con GROUP BY (Slide 11)

In [ ]:
# Slide 11: GROUP BY con funciones agregadas
df_group_by = pd.read_sql_query(
    """SELECT 
        region,
        COUNT(*) as total_ventas,
        ROUND(AVG(monto_total), 2) as promedio_ventas,
        MIN(monto_total) as venta_minima,
        MAX(monto_total) as venta_maxima,
        SUM(monto_total) as total_ingresos
    FROM ventas
    GROUP BY region
    ORDER BY total_ingresos DESC""",
    conn
)

print("SLIDE 11: GROUP BY con Funciones Agregadas")
print("\nEstadísticas por Región:")
print(df_group_by.to_string(index=False))

## 2. DISTINCT - Eliminar Duplicados (Slides 13-18)

In [ ]:
# Slide 14: DISTINCT básico
print("SLIDE 14: Sentencia DISTINCT")
print("="*70)

df_distinct_ciudad = pd.read_sql_query(
    'SELECT DISTINCT ciudad FROM ventas ORDER BY ciudad',
    conn
)

print("\nCiudades únicas donde hay ventas:")
for idx, row in df_distinct_ciudad.iterrows():
    print(f"  • {row['ciudad']}")

In [ ]:
# Slide 16: DISTINCT con múltiples columnas
df_distinct_multi = pd.read_sql_query(
    'SELECT DISTINCT region, ciudad FROM ventas ORDER BY region, ciudad',
    conn
)

print("\nSLIDE 16: DISTINCT con Múltiples Columnas")
print("\nCombinaciones únicas de Región y Ciudad:")
print(df_distinct_multi.to_string(index=False))

### Comparación DISTINCT vs GROUP BY (Slide 17)

In [ ]:
# Slide 17: Comparación
print("SLIDE 17: Comparación DISTINCT vs GROUP BY")
print("="*70)

# Con DISTINCT
df_distinct = pd.read_sql_query(
    'SELECT DISTINCT region FROM ventas',
    conn
)
print(f"\n1. DISTINCT region:")
print(f"   → Solo lista valores únicos (sin agregación)")
print(df_distinct.to_string(index=False))

# Con GROUP BY
df_group = pd.read_sql_query(
    'SELECT region, COUNT(*) as cantidad FROM ventas GROUP BY region',
    conn
)
print(f"\n2. GROUP BY region:")
print(f"   → Lista valores únicos + posibilidad de agregar funciones")
print(df_group.to_string(index=False))

## 3. OPERACIONES DE CONJUNTO (Slides 19-25)

### UNION - Combinar Resultados (Slide 20)

In [ ]:
# Slide 20: UNION
df_union = pd.read_sql_query(
    """SELECT nombre, correo, ciudad
    FROM clientes_2023
    UNION
    SELECT nombre, correo, ciudad
    FROM clientes_2024
    ORDER BY nombre""",
    conn
)

print("SLIDE 20: Operación UNION")
print("\nListado consolidado de clientes (sin duplicados):")
print(df_union.to_string(index=False))
print(f"\nTotal: {len(df_union)} clientes únicos")

### INTERSECT - Registros Comunes (Slide 22)

In [ ]:
# Slide 22: INTERSECT
df_intersect = pd.read_sql_query(
    """SELECT nombre
    FROM clientes_2023
    INTERSECT
    SELECT nombre
    FROM clientes_2024""",
    conn
)

print("SLIDE 22: Operación INTERSECT")
print("\nClientes presentes en AMBAS campañas (2023 y 2024):")
print(df_intersect.to_string(index=False))
print(f"\nTotal: {len(df_intersect)} clientes persistentes")

### EXCEPT - Registros Únicos (Slide 24)

In [ ]:
# Slide 24: EXCEPT
df_except = pd.read_sql_query(
    """SELECT nombre
    FROM clientes_2023
    EXCEPT
    SELECT nombre
    FROM clientes_2024""",
    conn
)

print("SLIDE 24: Operación EXCEPT")
print("\nClientes en 2023 que NO están en 2024:")
print(df_except.to_string(index=False))
print(f"\nTotal: {len(df_except)} clientes perdidos")

## 4. CASO PRÁCTICO: Reportes Consolidados (Slide 30)

In [ ]:
print("SLIDE 30: Consultas Guiadas - Caso Práctico")
print("="*70)

# Consulta 1: Ciudades distintas de ambas campañas
print("\n1️⃣ Obtener ciudades distintas de ambas campañas:")
df_1 = pd.read_sql_query(
    """SELECT DISTINCT ciudad 
    FROM clientes_2023
    UNION
    SELECT DISTINCT ciudad 
    FROM clientes_2024
    ORDER BY ciudad""",
    conn
)
print(df_1.to_string(index=False))

# Consulta 2: Ingreso promedio 2024
print("\n2️⃣ Calcular ingreso promedio 2024:")
df_2 = pd.read_sql_query(
    """SELECT 
        ROUND(AVG(ingreso_mensual), 2) as ingreso_promedio
    FROM clientes_2024""",
    conn
)
print(f"   Ingreso promedio: ${df_2.iloc[0, 0]:,.2f}")

# Consulta 3: Clientes en ambas campañas
print("\n3️⃣ Identificar clientes en ambas campañas:")
df_3 = pd.read_sql_query(
    """SELECT nombre
    FROM clientes_2023
    INTERSECT
    SELECT nombre
    FROM clientes_2024""",
    conn
)
print(df_3.to_string(index=False))

# Consulta 4: Clientes 2023 no presentes en 2024
print("\n4️⃣ Clientes 2023 que no están en 2024:")
df_4 = pd.read_sql_query(
    """SELECT nombre
    FROM clientes_2023
    EXCEPT
    SELECT nombre
    FROM clientes_2024""",
    conn
)
print(df_4.to_string(index=False))

## 5. ANÁLISIS COMPLETO: Reporte de Retail (Slide 3)

In [ ]:
print("SLIDE 3: Desafío Inicial - Reporte de Retail")
print("="*70)

# Requerimiento 1: Estadísticas por región
print("\n📊 Req 1: Promedio, máximo y mínimo de ventas por región")
df_req1 = pd.read_sql_query(
    """SELECT 
        region,
        COUNT(*) as total_transacciones,
        ROUND(AVG(monto_total), 2) as promedio_ventas,
        MIN(monto_total) as venta_minima,
        MAX(monto_total) as venta_maxima
    FROM ventas
    GROUP BY region
    ORDER BY promedio_ventas DESC""",
    conn
)
print(df_req1.to_string(index=False))

# Requerimiento 2: Conteo total
print("\n📊 Req 2: Conteo total de transacciones finalizadas")
df_req2 = pd.read_sql_query(
    """SELECT COUNT(*) as total_transacciones_finalizadas
    FROM ventas
    WHERE estado = 'Finalizada'""",
    conn
)
print(f"   Total: {df_req2.iloc[0, 0]} transacciones")

# Requerimiento 3: Listado sin duplicados
print("\n📊 Req 3: Listado de productos vendidos (sin duplicados)")
df_req3 = pd.read_sql_query(
    """SELECT DISTINCT producto
    FROM ventas
    ORDER BY producto""",
    conn
)
for idx, row in df_req3.iterrows():
    print(f"   • {row['producto']}")

## 6. BUENAS PRÁCTICAS Y ERRORES COMUNES

In [ ]:
buenas_practicas = """
    BUENAS PRÁCTICAS (Slides 6, 8, 10, 15)
    ═════════════════════════════════════════════════════════════
    
    MIN/MAX:
    ✓ Usar alias (AS) para claridad
    ✓ Aplicar sobre columnas indexadas
    
    AVG:
    ✓ Filtrar con WHERE antes de promediar
    ✓ Usar CASE WHEN para segmentar promedios
    ✓ Controlar NULL que puede distorsionar resultados
    
    COUNT:
    ✓ COUNT(*) para contar todas las filas
    ✓ COUNT(DISTINCT) para valores únicos
    ✓ COUNT(columna) excluye NULL
    
    DISTINCT:
    ✓ Validar que no hay errores de digitación
    ✓ Usar ORDER BY para mejorar legibilidad
    ✓ Combinar con otras funciones según necesidad
    
    UNION/INTERSECT/EXCEPT:
    ✓ Asegurar mismo número y tipo de columnas
    ✓ Validar que datos tienen formato compatible
    ✓ Usar UNION ALL si se desean duplicados
"""

print(buenas_practicas)

In [ ]:
errores_comunes = """
    ERRORES COMUNES A EVITAR
    ═════════════════════════════════════════════════════════════
    
    ❌ COUNT
    • COUNT(*) vs COUNT(columna) - distintos resultados
    • No considerar NULL excluye registros
    
    ❌ AVG
    • No filtrar valores atípicos
    • Ignorar que NULL no se cuenta
    
    ❌ DISTINCT
    • Aplicar sobre muchas columnas sin revisar
    • No controlar errores de digitación (Santiago vs santiago)
    
    ❌ UNION/INTERSECT/EXCEPT
    • Diferencia en tipos de dato entre SELECT
    • Diferente número de columnas
    • No considerar sensibilidad a mayúsculas en texto
    • Asumir que columnas tienen mismo nombre
    
    ❌ GROUP BY
    • Usar columnas no en GROUP BY sin función agregada
    • No revisar consistencia en datos (ej. regiones mal escritas)
    • Olvidar HAVING para filtrar grupos
"""

print(errores_comunes)

## RESUMEN (Slide 32)

In [ ]:
resumen = """
    SLIDE 32: RESUMEN DE SESIÓN 6
    ═════════════════════════════════════════════════════════════
    
    1️⃣  FUNCIONES AGREGADAS
        MIN(), MAX(), AVG(), COUNT()
        • Devuelven valores únicos (resumen)
        • Se usan con WHERE para filtrar
        • Se combinan con GROUP BY para análisis por grupos
    
    2️⃣  DISTINCT
        Elimina duplicados de resultados
        • DISTINCT columna: lista única
        • DISTINCT con múltiples columnas: combinaciones únicas
        • Vs GROUP BY: DISTINCT solo lista, GROUP BY permite agregar
    
    3️⃣  OPERACIONES DE CONJUNTO
        UNION: Combina y elimina duplicados
        INTERSECT: Solo registros en ambas consultas
        EXCEPT: Registros en primera pero no en segunda
        • Requieren mismo número y tipo de columnas
        • Útil para comparar tablas y períodos
    
    4️⃣  APLICACIONES REALES
        • Reportes consolidados
        • Análisis por región/categoría
        • Identificar cambios entre períodos
        • Limpiar datos y evitar duplicados
    
    5️⃣  PRÓXIMA SESIÓN
        Consultas sobre tablas relacionadas (JOINs)
        Claves primarias y foráneas
        Integridad referencial
"""

print(resumen)